<div align="center">

# 🔩 Steel Rotating Bending Fatigue Strength Prediction
### Predicting Fatigue Strength (10⁷ Cycles) using Machine Learning

[![Python](https://img.shields.io/badge/Python-3.8+-blue?logo=python)](https://python.org)
[![scikit-learn](https://img.shields.io/badge/scikit--learn-latest-orange?logo=scikit-learn)](https://scikit-learn.org)
[![XGBoost](https://img.shields.io/badge/XGBoost-latest-green)](https://xgboost.readthedocs.io)
[![Optuna](https://img.shields.io/badge/Optuna-HPO-blueviolet)](https://optuna.org)
[![SHAP](https://img.shields.io/badge/SHAP-Explainability-red)](https://shap.readthedocs.io)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](LICENSE)

</div>

---

## 📋 Project Overview

This notebook builds and evaluates **11 machine learning models** to predict the **rotating bending fatigue strength** of steel at 10⁷ cycles from a dataset of ~500 steel samples. It covers the full production ML pipeline:

| Step | Description |
|------|-------------|
| **EDA** | Distribution analysis, missing value detection |
| **Correlation Heatmap** | Feature–feature and feature–target relationships |
| **K-Means + PCA** | Unsupervised clustering & dimensionality reduction |
| **Model Training** | 11 regressors with proper scaling strategy |
| **Evaluation** | R², RMSE, MAE, 5-Fold Cross-Validation |
| **Hyperparameter Tuning** | Optuna Bayesian optimisation on the best model |
| **SHAP Explainability** | Global + local feature importance via SHAP values |
| **Prediction** | Predict fatigue for any custom steel sample |

## 🗂️ Dataset Features

**Processing Parameters:** `NT`, `THT`, `THt`, `THQCr`, `CT`, `Ct`, `DT`, `Dt`, `QmT`, `TT`, `Tt`, `TCr`, `RedRatio`, `dA`, `dB`, `dC`

**Chemical Composition:** `C`, `Si`, `Mn`, `P`, `S`, `Ni`, `Cr`, `Cu`, `Mo`

**Target:** `Fatigue` — Rotating Bending Fatigue Strength (MPa, 10⁷ cycles)

## 🤖 Models Used

| Category | Models |
|----------|--------|
| **Ensemble (Tree)** | Random Forest, Gradient Boosting, XGBoost |
| **Neural Network** | ANN (MLP) |
| **Bayesian** | Bayesian Ridge |
| **Regularized Linear** | Lasso, Ridge |
| **Linear** | Linear Regression |
| **Instance-Based** | K-Nearest Neighbors |
| **Tree** | Decision Tree |
| **Kernel** | Support Vector Regression |


---
## ⚙️ Section 1 — Environment Setup

In [ ]:
# Install external dependencies
!pip install xgboost optuna shap --quiet

# ── Core ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import io, os, warnings, time
warnings.filterwarnings('ignore')

# ── Visualisation ─────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Style
plt.rcParams.update({
    'figure.dpi'       : 120,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.25,
    'font.family'      : 'DejaVu Sans',
})
sns.set_palette('tab10')

# ── Sklearn — preprocessing & evaluation ──────────────
from sklearn.model_selection  import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing    import StandardScaler
from sklearn.metrics          import mean_squared_error, mean_absolute_error, r2_score
from sklearn.decomposition    import PCA
from sklearn.cluster          import KMeans

# ── Sklearn — models ──────────────────────────────────
from sklearn.ensemble         import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network   import MLPRegressor
from sklearn.linear_model     import BayesianRidge, Lasso, Ridge, LinearRegression
from sklearn.tree             import DecisionTreeRegressor
from sklearn.neighbors        import KNeighborsRegressor
from sklearn.svm              import SVR

# ── XGBoost ───────────────────────────────────────────
from xgboost import XGBRegressor

# ── Optuna (hyperparameter optimisation) ──────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── SHAP (model explainability) ───────────────────────
import shap

print('✅ All libraries loaded successfully.')
print(f'   NumPy   {np.__version__}')
print(f'   Pandas  {pd.__version__}')
print(f'   Optuna  {optuna.__version__}')
print(f'   SHAP    {shap.__version__}')


---
## 📂 Section 2 — Data Loading

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# OPTION A  ▸  Upload interactively (recommended for first-time use in Colab)
# ──────────────────────────────────────────────────────────────────────────────
from google.colab import files
print('📤 Please upload  fatigue_data_.csv  when the dialog appears …')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# ──────────────────────────────────────────────────────────────────────────────
# OPTION B  ▸  Mount Google Drive (comment-out Option A and uncomment below)
# ──────────────────────────────────────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/fatigue_data_.csv')

# ──────────────────────────────────────────────────────────────────────────────
# OPTION C  ▸  Local / already in working directory
# ──────────────────────────────────────────────────────────────────────────────
# df = pd.read_csv('fatigue_data_.csv')

print(f'\n✅ Dataset loaded  →  {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

---
## 🔍 Section 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1  Schema & basic info ─────────────────────────────────────────────────
print('─' * 55)
print('DATASET SCHEMA')
print('─' * 55)
df.info()

print('\n' + '─' * 55)
print('DESCRIPTIVE STATISTICS')
print('─' * 55)
df.describe().T.round(3)

In [ ]:
# ── 3.2  Missing-value audit ──────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
audit = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
audit = audit[audit['Missing Count'] > 0]

if audit.empty:
    print('✅ No missing values detected — dataset is complete.')
else:
    print('⚠️  Missing values found:')
    print(audit)
    df.dropna(inplace=True)
    print(f'\nRows after dropping NaNs: {len(df)}')

print(f'\nFinal dataset  →  {df.shape[0]} rows × {df.shape[1]} columns')

In [ ]:
# ── 3.3  Target variable (Fatigue) distribution ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Target Variable — Fatigue Strength Distribution', fontsize=14, fontweight='bold', y=1.01)

# Histogram
axes[0].hist(df['Fatigue'], bins=35, color='steelblue', edgecolor='white', alpha=0.88, linewidth=0.6)
axes[0].axvline(df['Fatigue'].mean(),   color='red',    linestyle='--', linewidth=1.8,
                label=f'Mean  = {df["Fatigue"].mean():.0f} MPa')
axes[0].axvline(df['Fatigue'].median(), color='orange', linestyle=':',  linewidth=1.8,
                label=f'Median = {df["Fatigue"].median():.0f} MPa')
axes[0].set_xlabel('Fatigue Strength (MPa)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Histogram', fontsize=12)
axes[0].legend()

# Box-plot
bp = axes[1].boxplot(
    df['Fatigue'], vert=True, patch_artist=True,
    widths=0.5,
    boxprops     = dict(facecolor='steelblue', color='navy', alpha=0.7),
    medianprops  = dict(color='red', linewidth=2.5),
    whiskerprops = dict(color='navy'),
    capprops     = dict(color='navy'),
    flierprops   = dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.5)
)
axes[1].set_ylabel('Fatigue Strength (MPa)', fontsize=11)
axes[1].set_title('Box-Plot', fontsize=12)
axes[1].set_xticks([])

plt.tight_layout()
plt.savefig('01_fatigue_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

stats = df['Fatigue'].describe()
print(f'  Min     : {stats["min"]:.0f} MPa')
print(f'  Max     : {stats["max"]:.0f} MPa')
print(f'  Mean    : {stats["mean"]:.1f} MPa')
print(f'  Std Dev : {stats["std"]:.1f} MPa')
print(f'  Skewness: {df["Fatigue"].skew():.3f}')

In [ ]:
# ── 3.4  Feature distributions (grid) ────────────────────────────────────────
feature_cols = [c for c in df.columns if c != 'Fatigue']
n_cols = 5
n_rows = -(-len(feature_cols) // n_cols)  # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 2.8))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].hist(df[col], bins=25, color='#457B9D', edgecolor='white', alpha=0.85, linewidth=0.4)
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🌡️ Section 4 — Correlation Heatmap

In [ ]:
corr = df.corr()

fig, axes = plt.subplots(1, 2, figsize=(22, 8),
                          gridspec_kw={'width_ratios': [3, 1]})

# ── Full heatmap (lower triangle) ────────────────────────────────────────────
mask_upper = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    ax         = axes[0],
    mask       = mask_upper,
    annot      = True,
    fmt        = '.2f',
    cmap       = 'RdYlGn',
    center     = 0,
    vmin       = -1, vmax = 1,
    linewidths = 0.4,
    linecolor  = 'white',
    annot_kws  = {'size': 6.5},
    square     = True,
    cbar_kws   = {'shrink': 0.75, 'label': 'Pearson r'}
)
axes[0].set_title('Feature Correlation Matrix  (lower triangle)', fontsize=13, fontweight='bold', pad=12)
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', rotation=0,  labelsize=8)

# ── Bar chart: top correlations with Fatigue ─────────────────────────────────
fatigue_corr = corr['Fatigue'].drop('Fatigue').sort_values(key=abs, ascending=True)
colors = ['#E63946' if v < 0 else '#2A9D8F' for v in fatigue_corr.values]
bars = axes[1].barh(fatigue_corr.index, fatigue_corr.values, color=colors, edgecolor='white', height=0.65)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Pearson r with Fatigue', fontsize=10)
axes[1].set_title('Correlation\nwith Fatigue', fontsize=11, fontweight='bold')
axes[1].set_xlim(-1, 1)
for bar, val in zip(bars, fatigue_corr.values):
    ha = 'left' if val >= 0 else 'right'
    offset = 0.02 if val >= 0 else -0.02
    axes[1].text(val + offset, bar.get_y() + bar.get_height() / 2,
                 f'{val:.2f}', va='center', ha=ha, fontsize=7)

plt.tight_layout()
plt.savefig('03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 features most correlated with Fatigue Strength:')
print(corr['Fatigue'].drop('Fatigue').sort_values(key=abs, ascending=False).head(10).round(3).to_string())

---
## 🔵 Section 5 — K-Means Clustering + PCA Visualisation

In [ ]:
# ── 5.1  Prepare & scale all features ────────────────────────────────────────
features    = [c for c in df.columns if c != 'Fatigue']
X_all       = df[features].values
y_all       = df['Fatigue'].values

scaler_full = StandardScaler()
X_scaled    = scaler_full.fit_transform(X_all)

# ── 5.2  Elbow method ────────────────────────────────────────────────────────
K_range  = range(2, 11)
inertias = []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    inertias.append(km.fit(X_scaled).inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, inertias, 'o-', color='steelblue', markersize=8, linewidth=2.2)
ax.set_xlabel('Number of Clusters  K', fontsize=12)
ax.set_ylabel('Inertia  (Within-Cluster SSE)', fontsize=12)
ax.set_title('Elbow Method — Optimal K Selection', fontsize=13, fontweight='bold')
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
plt.tight_layout()
plt.savefig('04_elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()
print('Inspect the elbow above and set N_CLUSTERS accordingly in the next cell.')

In [ ]:
# ── 5.3  K-Means fit ─────────────────────────────────────────────────────────
N_CLUSTERS = 4          # ← adjust based on the elbow plot above

CLUSTER_COLORS = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#F4A261', '#264653']

kmeans         = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# ── 5.4  PCA  →  2D ──────────────────────────────────────────────────────────
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
ev    = pca.explained_variance_ratio_

print(f'PCA  →  PC1 = {ev[0]*100:.1f}%  |  PC2 = {ev[1]*100:.1f}%  |  Total = {sum(ev)*100:.1f}%')

# ── 5.5  Plot ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('K-Means Clustering via PCA', fontsize=14, fontweight='bold', y=1.01)

# Left — cluster labels
for c in range(N_CLUSTERS):
    mask = cluster_labels == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=CLUSTER_COLORS[c], label=f'Cluster {c}  (n={mask.sum()})',
                    s=35, alpha=0.75, edgecolors='white', linewidths=0.3)
centers_2d = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centers_2d[:, 0], centers_2d[:, 1],
                c='black', marker='X', s=220, zorder=6, label='Centroid')
axes[0].set_xlabel(f'PC1  ({ev[0]*100:.1f}% variance)', fontsize=10)
axes[0].set_ylabel(f'PC2  ({ev[1]*100:.1f}% variance)', fontsize=10)
axes[0].set_title(f'Cluster Assignments  (K = {N_CLUSTERS})', fontsize=12)
axes[0].legend(fontsize=8)

# Right — fatigue strength gradient
sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                     c=y_all, cmap='plasma', s=35, alpha=0.85,
                     edgecolors='white', linewidths=0.2)
plt.colorbar(sc, ax=axes[1], label='Fatigue Strength (MPa)', shrink=0.85)
axes[1].set_xlabel(f'PC1  ({ev[0]*100:.1f}% variance)', fontsize=10)
axes[1].set_ylabel(f'PC2  ({ev[1]*100:.1f}% variance)', fontsize=10)
axes[1].set_title('Fatigue Strength Gradient', fontsize=12)

plt.tight_layout()
plt.savefig('05_kmeans_pca.png', dpi=150, bbox_inches='tight')
plt.show()

# Cluster summary table
tmp = df.copy()
tmp['Cluster'] = cluster_labels
print('\nCluster Summary — Fatigue Strength (MPa):')
print(tmp.groupby('Cluster')['Fatigue'].agg(['count','mean','std','min','max']).round(1).to_string())

---
## 🛠️ Section 6 — Data Preparation for Modelling

In [ ]:
X = df[features].copy()
y = df['Fatigue'].copy()

# ── Train / Test split  (80 / 20, stratify not needed for regression) ─────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# ── Standardise for models sensitive to feature scale ─────────────────────────
scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)   # fit ONLY on train
X_test_sc   = scaler.transform(X_test)        # transform test with same params

print('Data split summary')
print(f'  Training samples : {X_train.shape[0]}')
print(f'  Test     samples : {X_test.shape[0]}')
print(f'  Number of features: {X_train.shape[1]}')

---
## 🤖 Section 7 — Model Training & Evaluation

> **Scaling strategy:**  
> Tree-based models (Random Forest, Gradient Boosting, XGBoost, Decision Tree) are  
> trained on **raw** features. All other models receive **standardised** features.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Helper — fit, predict, evaluate
# ────────────────────────────────────────────────────────────────────────────
def train_and_evaluate(name, model, X_tr, y_tr, X_te, y_te, cv=5):
    """Fit model and return a metrics dict + predictions."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    return {
        'Model'         : name,
        'R²  (Test)'    : r2_score(y_te, y_pred),
        'RMSE (MPa)'    : np.sqrt(mean_squared_error(y_te, y_pred)),
        'MAE  (MPa)'    : mean_absolute_error(y_te, y_pred),
        'CV R²  (5-fold)': cross_val_score(model, X_tr, y_tr, cv=cv, scoring='r2').mean(),
        '_model'        : model,
        '_predictions'  : y_pred,
    }

# ────────────────────────────────────────────────────────────────────────────
# Model registry
# ────────────────────────────────────────────────────────────────────────────
MODEL_REGISTRY = {
    # ── Ensemble (tree-based) — do NOT standardise ──────────────────────────
    'Random Forest'      : RandomForestRegressor(
                               n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting'  : GradientBoostingRegressor(
                               n_estimators=200, learning_rate=0.05,
                               max_depth=4,      random_state=42),
    'XGBoost'            : XGBRegressor(
                               n_estimators=200, learning_rate=0.05,
                               max_depth=4,      random_state=42,
                               verbosity=0,      n_jobs=-1),
    'Decision Tree'      : DecisionTreeRegressor(
                               max_depth=8,      random_state=42),

    # ── Requires standardised input ──────────────────────────────────────────
    'ANN (MLP)'          : MLPRegressor(
                               hidden_layer_sizes=(128, 64, 32),
                               activation='relu',  max_iter=1000,
                               random_state=42,    early_stopping=True),
    'Bayesian Ridge'     : BayesianRidge(),
    'Lasso'              : Lasso(alpha=0.1, max_iter=10_000),
    'Ridge'              : Ridge(alpha=1.0),
    'Linear Regression'  : LinearRegression(),
    'KNN'                : KNeighborsRegressor(n_neighbors=7, n_jobs=-1),
    'SVR'                : SVR(kernel='rbf', C=100, epsilon=0.1, gamma='scale'),
}

TREE_MODELS = {'Random Forest', 'Gradient Boosting', 'XGBoost', 'Decision Tree'}

# ────────────────────────────────────────────────────────────────────────────
# Train all models
# ────────────────────────────────────────────────────────────────────────────
results_list  = []
model_store   = {}   # { name: fitted_model,  name+'_pred': np.array }

print(f'{"Model":<26}  {"R² Test":>9}  {"RMSE":>9}  {"MAE":>9}  {"CV R²":>9}')
print('─' * 70)

for name, model in MODEL_REGISTRY.items():
    Xtr = X_train    if name in TREE_MODELS else X_train_sc
    Xte = X_test     if name in TREE_MODELS else X_test_sc

    res                  = train_and_evaluate(name, model, Xtr, y_train, Xte, y_test)
    model_store[name]    = res.pop('_model')
    model_store[name+'_pred'] = res.pop('_predictions')
    results_list.append(res)

    print(f'  {name:<24}  {res["R²  (Test)"]:.4f}  {res["RMSE (MPa)"]:.2f}  '
          f'{res["MAE  (MPa)"]:.2f}  {res["CV R²  (5-fold)"]:.4f}')

# Build results DataFrame sorted by test R²
results_df = (
    pd.DataFrame(results_list)
    .sort_values('R²  (Test)', ascending=False)
    .reset_index(drop=True)
)
print('\n✅ All 11 models trained successfully.')

---
## 📊 Section 8 — Model Comparison

In [ ]:
# ── 8.1  Leaderboard table ────────────────────────────────────────────────────
display_df = results_df.copy()
display_df.index = display_df.index + 1   # rank starts at 1
display_df.index.name = 'Rank'
print('\n' + '='*70)
print('  MODEL LEADERBOARD  (sorted by R² on held-out test set)')
print('='*70)
print(display_df.to_string(float_format='%.4f'))

In [ ]:
# ── 8.2  Visual leaderboard ───────────────────────────────────────────────────
metrics   = ['R²  (Test)', 'RMSE (MPa)', 'CV R²  (5-fold)']
titles    = ['R²  Score (Test Set)', 'RMSE  (MPa)', 'Cross-Val R²  (5-Fold)']
xlabels   = ['R²', 'RMSE (MPa)', 'CV R²']
palettes  = ['RdYlGn', 'RdYlGn_r', 'RdYlGn']   # green = better

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle('Steel Fatigue Prediction — Model Performance Comparison',
             fontsize=15, fontweight='bold', y=1.02)

for ax, metric, title, xlabel, pal in zip(axes, metrics, titles, xlabels, palettes):
    colors = sns.color_palette(pal, len(results_df))
    bars   = ax.barh(results_df['Model'], results_df[metric],
                     color=colors, edgecolor='white', height=0.65)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.invert_yaxis()

    for bar, val in zip(bars, results_df[metric]):
        offset = bar.get_width() * 0.01
        ax.text(bar.get_width() + offset,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=8)

    if metric in ['R²  (Test)', 'CV R²  (5-fold)']:
        ax.axvline(0.90, color='navy', linestyle='--', alpha=0.55, linewidth=1.2, label='R²=0.90')
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('06_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.3  All-model Actual vs Predicted grid ───────────────────────────────────
all_preds = np.concatenate([model_store[n+'_pred'] for n in MODEL_REGISTRY])
g_min = min(y_test.min(), all_preds.min())
g_max = max(y_test.max(), all_preds.max())

COLS = 4
ROWS = -(-len(MODEL_REGISTRY) // COLS)
fig, axes = plt.subplots(ROWS, COLS, figsize=(COLS * 4.5, ROWS * 4.2))
axes = axes.flatten()

for i, (_, row) in enumerate(results_df.iterrows()):
    mname = row['Model']
    preds = model_store[mname + '_pred']
    r2    = row['R²  (Test)']
    ax    = axes[i]

    ax.scatter(y_test, preds, alpha=0.70, color='steelblue',
               edgecolors='white', linewidths=0.3, s=28)
    ax.plot([g_min, g_max], [g_min, g_max], 'r--', linewidth=1.4, label='Ideal')
    ax.set_xlim(g_min, g_max)
    ax.set_ylim(g_min, g_max)
    ax.set_title(f'{mname}\nR² = {r2:.4f}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Actual (MPa)', fontsize=8)
    ax.set_ylabel('Predicted (MPa)', fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle('Actual vs. Predicted — All Models', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('07_all_models_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Multi-panel plot saved.')

---
## 🏆 Section 9 — Best Model Deep Dive

In [ ]:
best_name  = results_df.iloc[0]['Model']
best_r2    = results_df.iloc[0]['R²  (Test)']
best_rmse  = results_df.iloc[0]['RMSE (MPa)']
best_preds = model_store[best_name + '_pred']
residuals  = y_test.values - best_preds

print(f'🏆 Best Model   : {best_name}')
print(f'   R² (Test)   : {best_r2:.4f}')
print(f'   RMSE        : {best_rmse:.2f} MPa')
print(f'   Residual σ  : {residuals.std():.2f} MPa')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Best Model Analysis — {best_name}  (R² = {best_r2:.4f})',
             fontsize=14, fontweight='bold', y=1.02)

# Actual vs Predicted
mn, mx = min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())
axes[0].scatter(y_test, best_preds, alpha=0.70, c='steelblue',
                edgecolors='white', linewidths=0.3, s=45)
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Ideal (y=x)')
axes[0].set_xlabel('Actual Fatigue (MPa)', fontsize=11)
axes[0].set_ylabel('Predicted Fatigue (MPa)', fontsize=11)
axes[0].set_title('Actual vs Predicted', fontsize=12)
axes[0].legend()

# Residuals vs Predicted
axes[1].scatter(best_preds, residuals, alpha=0.70, c='darkorange',
                edgecolors='white', linewidths=0.3, s=45)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.8)
axes[1].axhline( 2*residuals.std(), color='grey', linestyle=':', linewidth=1, label='±2σ')
axes[1].axhline(-2*residuals.std(), color='grey', linestyle=':', linewidth=1)
axes[1].set_xlabel('Predicted Fatigue (MPa)', fontsize=11)
axes[1].set_ylabel('Residual (MPa)', fontsize=11)
axes[1].set_title('Residual Plot', fontsize=12)
axes[1].legend(fontsize=8)

# Residual histogram
axes[2].hist(residuals, bins=28, color='mediumseagreen', edgecolor='white', alpha=0.88)
axes[2].axvline(0, color='red', linestyle='--', linewidth=1.8, label='Zero residual')
axes[2].set_xlabel('Residual (MPa)', fontsize=11)
axes[2].set_ylabel('Count', fontsize=11)
axes[2].set_title('Residual Distribution', fontsize=12)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('08_best_model_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📌 Section 10 — Feature Importance

In [ ]:
# Pick best tree-based model for feature importance
tree_names  = [n for n in TREE_MODELS if n in [r['Model'] for _, r in results_df.iterrows()]]
tree_subset = results_df[results_df['Model'].isin(TREE_MODELS)]
best_tree   = tree_subset.iloc[0]['Model']
tree_model  = model_store[best_tree]

importances = pd.Series(tree_model.feature_importances_, index=features).sort_values()

fig, ax = plt.subplots(figsize=(11, 8))
bar_colors = plt.cm.RdYlGn(np.linspace(0.15, 0.95, len(importances)))
bars = ax.barh(importances.index, importances.values, color=bar_colors, edgecolor='white', height=0.7)

for bar, val in zip(bars, importances.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8)

ax.set_xlabel('Feature Importance (Gain)', fontsize=12)
ax.set_title(f'Feature Importance — {best_tree}', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('09_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Top 10 most important features ({best_tree}):')
print(importances.sort_values(ascending=False).head(10).round(4).to_string())

---
## 🎯 Section 11 — Hyperparameter Tuning with Optuna

> **Optuna** uses **Tree-structured Parzen Estimator (TPE)** — a Bayesian strategy that
> learns from previous trials to propose better hyperparameter sets, far more efficient
> than exhaustive GridSearchCV on large search spaces.

The best-performing model from Section 7 is automatically selected for tuning.
After optimisation the tuned model is re-evaluated and compared against the baseline.


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────N_TRIALS   = 80    # increase for better optimisation (costs more time)CV_FOLDS   = 5TIMEOUT    = None  # set e.g. 300 (seconds) to cap total search timeprint(f'🔍 Tuning model : {best_name}')print(f'   Trials       : {N_TRIALS}')print(f'   CV folds     : {CV_FOLDS}')print()# ── Define the search space for each supported model ─────────────────────────def make_objective(model_name, X_tr, y_tr):    def objective(trial):        if model_name == 'XGBoost':            params = dict(                n_estimators      = trial.suggest_int('n_estimators', 100, 800),                learning_rate     = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),                max_depth         = trial.suggest_int('max_depth', 3, 10),                subsample         = trial.suggest_float('subsample', 0.5, 1.0),                colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),                reg_alpha         = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),                reg_lambda        = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),                min_child_weight  = trial.suggest_int('min_child_weight', 1, 10),                gamma             = trial.suggest_float('gamma', 0, 5),                random_state=42, verbosity=0, n_jobs=-1,            )            model = XGBRegressor(**params)        elif model_name == 'Random Forest':            params = dict(                n_estimators      = trial.suggest_int('n_estimators', 100, 800),                max_depth         = trial.suggest_int('max_depth', 3, 25),                min_samples_split = trial.suggest_int('min_samples_split', 2, 20),                min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 10),                max_features      = trial.suggest_categorical('max_features', ['sqrt','log2', 0.5, 0.8]),                bootstrap         = trial.suggest_categorical('bootstrap', [True, False]),                random_state=42, n_jobs=-1,            )            model = RandomForestRegressor(**params)        elif model_name == 'Gradient Boosting':            params = dict(                n_estimators      = trial.suggest_int('n_estimators', 100, 600),                learning_rate     = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),                max_depth         = trial.suggest_int('max_depth', 2, 8),                subsample         = trial.suggest_float('subsample', 0.5, 1.0),                min_samples_split = trial.suggest_int('min_samples_split', 2, 20),                min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 10),                max_features      = trial.suggest_categorical('max_features', ['sqrt','log2', None]),                random_state=42,            )            model = GradientBoostingRegressor(**params)        else:  # generic fallback — RF            params = dict(                n_estimators=trial.suggest_int('n_estimators', 100, 500),                max_depth=trial.suggest_int('max_depth', 3, 20),                random_state=42, n_jobs=-1,            )            model = RandomForestRegressor(**params)        score = cross_val_score(model, X_tr, y_tr, cv=CV_FOLDS,                                scoring='r2', n_jobs=-1).mean()        return score    return objective# ── Run optimisation ─────────────────────────────────────────────────────────Xtr_tune = X_train if best_name in TREE_MODELS else X_train_scXte_tune = X_test  if best_name in TREE_MODELS else X_test_scstudy = optuna.create_study(direction='maximize',                             sampler=optuna.samplers.TPESampler(seed=42))t0 = time.time()study.optimize(make_objective(best_name, Xtr_tune, y_train),               n_trials=N_TRIALS, timeout=TIMEOUT, show_progress_bar=True)elapsed = time.time() - t0print(f'\n✅ Optimisation complete  ({elapsed:.0f} s)')print(f'   Best CV R²  : {study.best_value:.4f}')print(f'   Best params :')for k, v in study.best_params.items():    print(f'     {k:<25} = {v}')

In [ ]:
# ── Retrain tuned model on full training set ────────────────────────────────best_params = study.best_paramsif best_name == 'XGBoost':    tuned_model = XGBRegressor(**best_params, random_state=42, verbosity=0, n_jobs=-1)elif best_name == 'Random Forest':    tuned_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)elif best_name == 'Gradient Boosting':    tuned_model = GradientBoostingRegressor(**best_params, random_state=42)else:    tuned_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)tuned_model.fit(Xtr_tune, y_train)tuned_preds = tuned_model.predict(Xte_tune)tuned_r2   = r2_score(y_test, tuned_preds)tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_preds))tuned_mae  = mean_absolute_error(y_test, tuned_preds)baseline_r2   = results_df.iloc[0]['R²  (Test)']baseline_rmse = results_df.iloc[0]['RMSE (MPa)']print('\n' + '='*55)print('  TUNING RESULT COMPARISON')print('='*55)print(f'  {"Metric":<15}  {"Baseline":>12}  {"Tuned":>12}  {"Delta":>10}')print('─'*55)print(f'  {"R² (Test)":<15}  {baseline_r2:>12.4f}  {tuned_r2:>12.4f}  '      f'{tuned_r2 - baseline_r2:>+10.4f}')print(f'  {"RMSE (MPa)":<15}  {baseline_rmse:>12.2f}  {tuned_rmse:>12.2f}  '      f'{tuned_rmse - baseline_rmse:>+10.2f}')print(f'  {"MAE (MPa)":<15}  {results_df.iloc[0]["MAE  (MPa)"]:>12.2f}  {tuned_mae:>12.2f}  '      f'{tuned_mae - results_df.iloc[0]["MAE  (MPa)"]:>+10.2f}')# ── Plot 1: Optimisation history ─────────────────────────────────────────────trial_nums  = [t.number for t in study.trials]trial_vals  = [t.value  for t in study.trials]running_max = [max(trial_vals[:i+1]) for i in range(len(trial_vals))]fig, axes = plt.subplots(1, 2, figsize=(16, 5))fig.suptitle(f'Optuna Hyperparameter Optimisation — {best_name}',             fontsize=14, fontweight='bold', y=1.01)# Historyaxes[0].scatter(trial_nums, trial_vals, alpha=0.45, s=22,                color='steelblue', label='Trial R²')axes[0].plot(trial_nums, running_max, color='red', linewidth=2,             label='Running best')axes[0].set_xlabel('Trial Number', fontsize=11)axes[0].set_ylabel('CV R²', fontsize=11)axes[0].set_title('Optimisation History', fontsize=12, fontweight='bold')axes[0].legend()# Baseline vs Tuned comparison barlabels  = ['Baseline', 'Tuned']r2_vals = [baseline_r2, tuned_r2]colors  = ['#457B9D', '#E63946']bars = axes[1].bar(labels, r2_vals, color=colors, width=0.45,                   edgecolor='white', linewidth=1.2)for bar, val in zip(bars, r2_vals):    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.005,                 f'{val:.4f}', ha='center', va='top',                 fontsize=13, fontweight='bold', color='white')axes[1].set_ylabel('R² Score (Test)', fontsize=11)axes[1].set_title('Baseline vs Tuned R²', fontsize=12, fontweight='bold')axes[1].set_ylim(max(0, min(r2_vals) - 0.02), 1.0)plt.tight_layout()plt.savefig('10_optuna_tuning.png', dpi=150, bbox_inches='tight')plt.show()# ── Plot 2: Param importances (Optuna built-in) ───────────────────────────────try:    param_imp = optuna.importance.get_param_importances(study)    names_imp = list(param_imp.keys())    vals_imp  = list(param_imp.values())    fig2, ax2 = plt.subplots(figsize=(9, max(4, len(names_imp)*0.55)))    cols_imp  = plt.cm.RdYlGn(np.linspace(0.15, 0.95, len(names_imp)))[::-1]    bars2 = ax2.barh(names_imp[::-1], vals_imp[::-1],                     color=cols_imp, edgecolor='white', height=0.65)    for bar, val in zip(bars2, vals_imp[::-1]):        ax2.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,                 f'{val:.3f}', va='center', fontsize=9)    ax2.set_xlabel('Importance (Fanova)', fontsize=11)    ax2.set_title(f'Hyperparameter Importance — {best_name}', fontsize=13, fontweight='bold')    plt.tight_layout()    plt.savefig('11_optuna_param_importance.png', dpi=150, bbox_inches='tight')    plt.show()except Exception as e:    print(f'Param importance plot skipped: {e}')# Keep tuned model + predictions for SHAPprint('\n✅ Tuned model ready for SHAP analysis in the next section.')

---
## 🔬 Section 12 — SHAP Model Explainability

> **SHAP (SHapley Additive exPlanations)** assigns each feature a fair contribution
> to every individual prediction based on cooperative game theory.
> Unlike simple feature importance, SHAP shows *direction* (positive / negative effect)
> and works for *any* model.

This section generates **5 SHAP visualisations** on the tuned best model:

| Plot | What it shows |
|------|---------------|
| **Summary (beeswarm)** | Global feature ranking + direction + magnitude |
| **Bar chart** | Mean absolute SHAP — clean ranking |
| **Dependence plots** | How top features interact with fatigue |
| **Heatmap** | SHAP values across all test samples |
| **Waterfall (single sample)** | Local explanation for one prediction |


In [ ]:
# ── Build SHAP explainer ─────────────────────────────────────────────────────# TreeExplainer is exact and very fast for tree-based modelsexplainer   = shap.TreeExplainer(tuned_model)X_test_df   = pd.DataFrame(Xte_tune, columns=features)   # keep feature namesshap_values = explainer(X_test_df)                        # Explanation objectprint(f'SHAP values computed for {X_test_df.shape[0]} test samples × {X_test_df.shape[1]} features.')print(f'SHAP array shape: {shap_values.values.shape}')# ── Plot 1: Beeswarm / Summary ────────────────────────────────────────────────plt.figure(figsize=(11, 8))shap.plots.beeswarm(shap_values, max_display=25, show=False)plt.title('SHAP Summary Plot (Beeswarm)\n'          'Each dot = one test sample  |  Colour = feature value  |  x-axis = impact on prediction',          fontsize=12, fontweight='bold', pad=12)plt.tight_layout()plt.savefig('12_shap_beeswarm.png', dpi=150, bbox_inches='tight')plt.show()print('\n🔑 Features on the right (positive SHAP) push prediction UP.')print('   Features on the left (negative SHAP) push prediction DOWN.')

In [ ]:
# ── Plot 2: Global bar chart (mean |SHAP|) ───────────────────────────────────plt.figure(figsize=(10, 7))shap.plots.bar(shap_values, max_display=25, show=False)plt.title('Mean Absolute SHAP Values — Global Feature Importance',          fontsize=13, fontweight='bold', pad=10)plt.tight_layout()plt.savefig('13_shap_bar.png', dpi=150, bbox_inches='tight')plt.show()

In [ ]:
# ── Plot 3: Dependence plots for top-3 features ──────────────────────────────mean_abs_shap = np.abs(shap_values.values).mean(axis=0)top3_idx      = np.argsort(mean_abs_shap)[::-1][:3]top3_features = [features[i] for i in top3_idx]fig, axes = plt.subplots(1, 3, figsize=(18, 5))fig.suptitle('SHAP Dependence Plots — Top 3 Features',             fontsize=14, fontweight='bold', y=1.02)for ax, feat in zip(axes, top3_features):    shap.dependence_plot(        feat,        shap_values.values,        X_test_df,        ax=ax,        show=False,        alpha=0.7,        dot_size=25,    )    ax.set_title(f'Dependence: {feat}', fontsize=11, fontweight='bold')plt.tight_layout()plt.savefig('14_shap_dependence.png', dpi=150, bbox_inches='tight')plt.show()print(f'Top 3 influential features: {top3_features}')

In [ ]:
# ── Plot 4: SHAP heatmap (all samples × all features) ───────────────────────plt.figure(figsize=(14, 8))shap.plots.heatmap(shap_values, max_display=20, show=False)plt.title('SHAP Heatmap — Feature Contributions Across All Test Samples',          fontsize=13, fontweight='bold', pad=10)plt.tight_layout()plt.savefig('15_shap_heatmap.png', dpi=150, bbox_inches='tight')plt.show()

In [ ]:
# ── Plot 5: Waterfall — local explanation for selected samples ───────────────# Identify the sample closest to the median fatigue strength (a 'typical' prediction)median_val   = np.median(y_test.values)median_idx   = np.argmin(np.abs(tuned_preds - median_val))# Also identify the prediction with the largest error (most interesting to explain)errors       = np.abs(y_test.values - tuned_preds)worst_idx    = np.argmax(errors)for label, idx in [('Median Prediction (typical sample)', median_idx),                   ('Largest Error Sample', worst_idx)]:    fig = plt.figure(figsize=(12, 6))    shap.plots.waterfall(shap_values[idx], max_display=15, show=False)    actual = y_test.values[idx]    pred   = tuned_preds[idx]    plt.title(f'SHAP Waterfall — {label}\n'              f'Actual: {actual:.1f} MPa  |  Predicted: {pred:.1f} MPa  |  Error: {actual-pred:+.1f} MPa',              fontsize=11, fontweight='bold')    plt.tight_layout()    safe_name = label.replace(' ','_').replace('(','').replace(')','').lower()    plt.savefig(f'16_shap_waterfall_{safe_name}.png', dpi=150, bbox_inches='tight')    plt.show()print('\n✅ All SHAP plots generated.')print('   → Features above the base value push the prediction UP   (red)')print('   → Features below the base value push the prediction DOWN  (blue)')

---
## 🔮 Section 13 — Predict on a New Steel Sample

The **tuned** best model is used for prediction here.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fill in the values for your new steel sample below.
# All 25 feature columns must be provided.
# The TUNED best model is used for the final prediction.
# ─────────────────────────────────────────────────────────────────────────────
NEW_SAMPLE = {
    # Processing parameters
    'NT'      : 885,    'THT'     : 30,    'THt'    : 0,     'THQCr'  : 0,
    'CT'      : 30,     'Ct'      : 0,     'DT'     : 30,    'Dt'     : 0,
    'QmT'     : 30,     'TT'      : 30,    'Tt'     : 0,     'TCr'    : 0,
    'RedRatio': 825,    'dA'      : 0.07,  'dB'     : 0.02,  'dC'     : 0.04,
    # Chemical composition (%)
    'C'       : 0.26,   'Si'      : 0.21,  'Mn'     : 0.44,  'P'      : 0.017,
    'S'       : 0.022,  'Ni'      : 0.01,  'Cr'     : 0.02,  'Cu'     : 0.01,
    'Mo'      : 0.00,
}

new_df = pd.DataFrame([NEW_SAMPLE])[features]   # enforce column order
new_inp = new_df if best_name in TREE_MODELS else scaler.transform(new_df)

# ── Baseline models ───────────────────────────────────────────────────────────
print('All baseline model predictions:')
print('─' * 42)
for name in MODEL_REGISTRY:
    m   = model_store[name]
    inp = new_df if name in TREE_MODELS else scaler.transform(new_df)
    print(f'  {name:<26} →  {m.predict(inp)[0]:.1f} MPa')

# ── Tuned model ───────────────────────────────────────────────────────────────
tuned_pred_new = tuned_model.predict(new_inp)[0]
print(f'\n🏆 Tuned {best_name} (Optuna) →  {tuned_pred_new:.1f} MPa')

# ── SHAP waterfall for new sample ─────────────────────────────────────────────
shap_new = explainer(pd.DataFrame([NEW_SAMPLE])[features])
plt.figure(figsize=(12, 6))
shap.plots.waterfall(shap_new[0], max_display=15, show=False)
plt.title(f'SHAP Explanation for New Sample\n'
          f'Predicted Fatigue Strength: {tuned_pred_new:.1f} MPa',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('17_shap_new_sample.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 💾 Section 14 — Save All Outputs


In [ ]:
# ── Save model results CSV ────────────────────────────────────────────────────
results_df.to_csv('model_results.csv', index=False)
print('✅ Saved: model_results.csv')

# ── Download everything to local machine ──────────────────────────────────────
OUTPUT_FILES = [
    'model_results.csv',
    '01_fatigue_distribution.png',
    '02_feature_distributions.png',
    '03_correlation_heatmap.png',
    '04_elbow_method.png',
    '05_kmeans_pca.png',
    '06_model_comparison.png',
    '07_all_models_actual_vs_predicted.png',
    '08_best_model_analysis.png',
    '09_feature_importance.png',
    '10_optuna_tuning.png',
    '11_optuna_param_importance.png',
    '12_shap_beeswarm.png',
    '13_shap_bar.png',
    '14_shap_dependence.png',
    '15_shap_heatmap.png',
    '16_shap_waterfall_median_prediction_typical_sample.png',
    '16_shap_waterfall_largest_error_sample.png',
]

from google.colab import files
import os

for f in OUTPUT_FILES:
    if os.path.exists(f):
        files.download(f)
        print(f'  ⬇️  {f}')
    else:
        print(f'  ⚠️  Not found: {f}')

print('\n✅ All outputs downloaded.')
